# Dice payouts

Two independent fair six-sided dice are rolled, and the contract pays $X = \max(D_1, D_2)$ euros. The dealer buys at €4.20 and sells at €4.60; our eventual aim is to choose between buying, selling, and doing nothing to maximise expected profit.

&nbsp;

For $ k \in \{ 1, 2, 3, 4, 5, 6 \} $ we have

$$ P (X \leq k) = P (D_1 \leq k,~ D_2 \leq k) = P (D_1 \leq k)P (D_2 \leq k) $$

and 

$$  P (D_i \leq k) = \frac{k}{6} \implies P (X \leq k) = \frac{k^2}{36}. $$

From that we can write

$$ P (X=k) =  P (X \leq k) - P (X < k) = P (X \leq k) - P (X \leq (k-1))$$

&nbsp;

$$ P (X=k) = \frac{2k-1}{36},~~~\mathbb{E}[X] = \frac{161}{36} $$

In [45]:
import numpy as np

#Two independent six faced fair die

values = np.arange(1, 7)
column = values.reshape(6, 1)
row = values.reshape(1, 6)
maxima = np.maximum(column, row)


prob = np.count_nonzero(maxima == 3)/36
sum_prob = 0

for k in values:
    sum_prob += np.count_nonzero(maxima == k)/36
    print(k, np.count_nonzero(maxima == k)/36)

np.mean(maxima)
delta_mean = (161/36) - np.mean(maxima)


seller_profit = 4.20 - maxima
buyer_profit = maxima - 4.60

mean_seller = np.mean(seller_profit)
mean_buyer = np.mean(buyer_profit)

assert np.isclose(1, sum_prob) #Probabilities must sum up to 1
assert np.isclose(161/36, np.mean(maxima))


print(mean_seller, mean_buyer)
print(f"The sum of probabilities is {sum_prob: .0f}")

1 0.027777777777777776
2 0.08333333333333333
3 0.1388888888888889
4 0.19444444444444445
5 0.25
6 0.3055555555555556
-0.27222222222222203 -0.12777777777777743
The sum of probabilities is  1


The action that maximises the expected profit is doing nothing, since that will have an expected profits of $ € 0 $ which is larger than the seller and buyer expected profits. The buyer can still receive a positive payout if the roll of the die is for instance 3 and 4 but on the long run he will lose money.

In [ ]:
#Two independent fair dice, each with n faces numbered 1,...,n

def dice_maxima(n_faces):
    values = np.arange(1, n_faces + 1)
    column = values.reshape(n_faces, 1)
    row = values.reshape(1, n_faces)
    maxima = np.maximum(column, row)

    return maxima

assert np.array_equal(dice_maxima(1), np.array([[1]]))
assert np.array_equal(dice_maxima(2), np.array([[1, 2],[2,2]]))

In [ ]:
#Simulation of #n fair dice

n = 4000

rng = np.random.default_rng(42)
rolls = rng.integers(1, 7, size=(n, 2))
rolls[:5]
rolls.shape

simulated_maxima = np.max(rolls, axis=1)
simulated_maxima.shape
simulated_mean = np.mean(simulated_maxima)
simulated_delta = 161/36 - np.mean(simulated_maxima)
estimated_se = np.std(simulated_maxima, ddof=1)/ np.sqrt(simulated_maxima.size)

discrepancy = (simulated_mean - 161/36)/ estimated_se

upper_bound = simulated_mean + 1.96 * estimated_se
lower_bound = simulated_mean - 1.96 * estimated_se


print(f"Estimated mean: €{simulated_mean:.4f}")
print(f"Actual discrepency: €{simulated_delta:.4f}")
print(f"Estimated standard error: €{estimated_se:.4f}")
print(f"The estimated mean is {discrepancy: .4f} standard errors different the exact expectation")

if lower_bound <= 161/36 <= upper_bound:
    print(f"The interval contains the benchmark")
else:
    print(f"The interval does not contain the benchmark")


Estimated mean: €4.4367
Actual discrepency: €0.0355
Estimated standard error: €0.0224
The estimated mean is -1.5812 standard errors different the exact expectation
The interval contains the benchmark


In [91]:
#Running multiple dice experiments simultaneuously 

rng = np.random.default_rng(42)

n_experiments = 200
covered = 0

for i in range(n_experiments):
    rolls = rng.integers(1, 7, size=(n, 2))
    
    simulated_maxima = np.max(rolls, axis=1)
    simulated_mean = np.mean(simulated_maxima)
    estimated_se = np.std(simulated_maxima, ddof=1)/ np.sqrt(simulated_maxima.size)
    
    upper_bound = simulated_mean + 1.96 * estimated_se
    lower_bound = simulated_mean - 1.96 * estimated_se

    if lower_bound <= 161/36 <= upper_bound:
        covered += 1


observed_coverage = covered / n_experiments

print(f"The proportion of experiments whose confidence interval contained the true expectation {observed_coverage * 100} %")

The proportion of experiments whose confidence interval contained the true expectation 97.0 %


We performed a run of 200 experiments, from those a seperate mean and estimated standard error were calculated. For a 95% confindence interval centered around the mean 194 out of the 200 runs contained the true expectation $(161/36)$ inside them, the rest did not. The remaining six misses do not mean that there was something wrong with the simulation code. It just means that our random sampling shifted those six sample means far enough from the truth that their intervals missed it. Increasing the number of trials (not to be confused with number of experiments) narrows down the interval since it decreases the estimated error but that does not guarantee that every interval will contain the truth since the interval is centered around the estimated mean.